# Object Detection Pipeline - Hyperparameter Tuning & Model Training
This notebook demonstrates the complete pipeline for training different YOLO models on aerial imagery. Besides, Hyperparameter tuning with Optuma, followed by final training and evaluation on test set.

Github Repo and Documentation of the work : [DL4CV Coconut Detection](https://github.com/kshitijrajsharma/dl4cv-oda)

By/ Kshitij Raj Sharma, Sahar Mohamed

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/kshitijrajsharma/dl4cv-oda/blob/master/notebooks/pipeline.ipynb)

This dl4cv_oda package includes all the pipline steps and functions for coconut trees, more info in the repo here: [DL4CV Coconut Detection](https://github.com/kshitijrajsharma/dl4cv-oda)

In [1]:
# ! pip install dl4cv_oda

# Object Detection Models Summary

## Comparison Table

| Model | Type | Key Architecture | Main Innovation | Strengths | Use Case |
|-------|------|-----------------|----------------|-----------|----------|
| **YOLOv8** | CNN-based | Backbone + Neck (FPN/PAN) + Split Head | Anchor-free, C2f modules | High speed, multi-task support | Real-time detection, balanced speed/accuracy |
| **YOLOv12** | CNN + Attention | R-ELAN Backbone + Area Attention | Attention mechanisms in YOLO | Better context, small object detection | Real-time with enhanced accuracy |
| **RT-DETR** | Transformer | Hybrid Encoder + Query Selection | End-to-end, NMS-free | Crowded scenes, global context | Complex scenes, research applications |

## YOLOv8

**Architecture:** Backbone → Neck → Head

**Key Features:**
- Anchor-free detection (direct center prediction)
- C2f modules (replaces C3 blocks)
- Decoupled classification/regression heads
- Multi-task:  detection, segmentation, classification, pose

**Best for:** General-purpose real-time detection

## YOLOv12

**Architecture:** R-ELAN Backbone + Area Attention Module

**Key Features:**
- Area Attention for high-res feature maps
- Residual Efficient Layer Aggregation Networks (R-ELAN)
- Attention-friendly architecture
- Optimized gradient flow

**Best for:** Small/detailed objects with real-time constraints

## RT-DETR

**Architecture:** Hybrid Encoder + Uncertainty-Minimal Query Selection

**Key Features:**
- First real-time Transformer detector
- End-to-end (no NMS, no anchors)
- Hybrid multi-scale encoder
- Fixed set object prediction

**Best for:** Dense/crowded scenes, GPU deployment

**Source:** Ultralytics

In [2]:
import requests
import geopandas as gpd
import json
import yaml
import time
import torch
import optuna
import pandas as pd
from pathlib import Path
from datetime import datetime
from ultralytics import YOLO, RTDETR
from dl4cv_oda import (clean_osm_data, clip_labels_to_tiles, convert_to_yolo_format,
                       create_train_val_split, create_yolo_config, download_tiles)

## Step 1: Data Preprocessing

In [ ]:
DATA_DIR = Path.cwd().parent / "data"
RAW_DIR = DATA_DIR / "raw"
CHIPS_DIR = DATA_DIR / "chips"
LABELS_DIR = DATA_DIR / "labels"
YOLO_DIR = DATA_DIR / "yolo"

TARGET = 'Coconut' # we decided to focus only on coconut trees as labels for other trees type were very low , original distribution : Coconut trees: 10,092, Mango: 261, Banana: 181, Papaya: 97

OSM_FILE = RAW_DIR / "kolovai-trees.geojson" # original osm data
CLEANED_FILE = RAW_DIR / "cleaned.geojson" ## cleaned osm data with only coconut trees and null value dropped & species mapping
TREES_BOX_FILE = DATA_DIR / "trees_box.geojson" # bounding boxes around each tree
TILES_FILE = DATA_DIR / "tiles.geojson" # patching the drone imagery into tiles

if not OSM_FILE.exists():
    OSM_FILE.parent.mkdir(parents=True, exist_ok=True)
    OSM_FILE.write_bytes(requests.get("https://github.com/kshitijrajsharma/dl4cv-oda/blob/master/data/raw/kolovai-trees.geojson? raw=true", allow_redirects=True).content)
    print(f"Downloaded OSM data")

if not CLEANED_FILE.exists():
    count = clean_osm_data(str(OSM_FILE), str(CLEANED_FILE), str(TREES_BOX_FILE),target=TARGET)
    print(f"Cleaned {count} trees")

if not TILES_FILE.exists():
    data = gpd.read_file(TREES_BOX_FILE)
    data. to_crs(epsg=4326, inplace=True)
    bbox = list(data.total_bounds)
    await download_tiles(bbox, 19, "https://tiles.openaerialmap.org/5a28639331eff4000c380690/0/5b1b6fb2-5024-4681-a175-9b667174f48c/{z}/{x}/{y}.png", DATA_DIR, 'OAM')
    print("Downloaded tiles")

label_stats = {}
if not (YOLO_DIR / "train").exists():
    label_stats = clip_labels_to_tiles(str(TREES_BOX_FILE), str(TILES_FILE), str(LABELS_DIR))
    print(f"Clipped labels to tiles: {label_stats}")
    
    # Convert our geojson labels to YOLO format
    class_mapping = convert_to_yolo_format(str(TREES_BOX_FILE), str(CHIPS_DIR), str(LABELS_DIR), str(YOLO_DIR))
    print(f"Converted to YOLO format")
    
    # Do spatial train/val/test split ( 70 / 20 / 10 % )
    train_count, val_count, test_count = create_train_val_split(str(LABELS_DIR), str(CHIPS_DIR), str(YOLO_DIR), train_ratio=0.7, val_ratio=0.2, test_ratio=0.1, seed=42)
    print(f"Split:  train={train_count}, val={val_count}, test={test_count}")
    
    # Create YOLO config file with our class mapping
    config_file = create_yolo_config(str(YOLO_DIR), {"Coconut": 0})
    print(f"Config:  {config_file}")

print("Data preparation complete")

Data preparation complete


## Step 2: Configuration

In [ ]:

RESULTS_DIR = Path("results")
RESULTS_DIR.mkdir(exist_ok=True)

# Configuration for training
SEED = 64
IMG_SIZE = 256
EPOCHS = 200
PATIENCE = 30 # early stopping patience , if training metric does not improve for these many epochs , training stops
BATCH = 16

# Configuration for hyperparameter tuning
TUNE_DEFAULT = False
TUNE_OPTUNA = True
TUNE_ITERATIONS = 6
TUNE_EPOCHS = 60
TUNE_PATIENCE = 10

# Models to train , here large variants of YOLOv8 , YOLOv12 and RTDETR are used for consistency in comparison
MODELS = [
    {"name": "yolov8l", "weights": "yolov8l.pt"},
    {"name": "yolo12l", "weights": "yolo12l.pt"},
    {"name": "rtdetr-l", "weights": "rtdetr-l.pt"},
]

EXPERIMENT_NAME = "full_pipeline"


torch.manual_seed(SEED)
exp_id = datetime.now().strftime("%Y%m%d_%H%M%S")
EXPERIMENT_NAME = f"{EXPERIMENT_NAME}_{exp_id}" if EXPERIMENT_NAME else exp_id
print(f"Experiment:  {EXPERIMENT_NAME}")
print(f"Models: {[m['name'] for m in MODELS]}")
print(f"Device: {torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'CPU'}")
print(f"Device Memory: {torch.cuda.get_device_properties(0).total_memory / (1024.0 **3):.2f} GB" if torch.cuda.is_available() else "N/A")


Experiment:  full_pipeline_20260113_225000
Models: ['yolov8l', 'yolo12l', 'rtdetr-l']
Device: NVIDIA GeForce RTX 4090 Laptop GPU
Device Memory: 15.57 GB


## Helper Functions

In [ ]:
def calculate_metrics(metrics):
    p, r = float(metrics. box. mp), float(metrics.box.mr)
    f1 = 2 * (p * r) / (p + r + 1e-6)
    return {
        'precision': p,
        'recall': r,
        'f1': f1,
        'map50': float(metrics.box.map50),
        'map50_95': float(metrics.box.map)
    }

def train_and_evaluate(model, name, run_name, hyperparams=None):
    start_time = time.time()
    
    train_params = {
        'data':  str(YOLO_DIR / "config.yaml"),
        'epochs': EPOCHS,
        'imgsz': IMG_SIZE,
        'patience': PATIENCE,
        'batch': BATCH,
        'seed': SEED,
        'name': run_name,
        'project': 'runs',
        'plots': True,
        'verbose': False,
    }
    
    if hyperparams:
        train_params.update(hyperparams)
    
    model.train(**train_params)
    train_time = time.time() - start_time
    
    val_start = time.time()
    val_metrics = model.val(split='val', verbose=False)
    val_time = time.time() - val_start
    
    test_start = time.time()
    test_metrics = model.val(split='test', verbose=False)
    test_time = time.time() - test_start
    
    return {
        'val':  calculate_metrics(val_metrics),
        'test': calculate_metrics(test_metrics),
        'train_time': train_time,
        'val_inference_time': val_time,
        'test_inference_time': test_time
    }

def tune_with_optuna(model_cfg, name):
    #Tune lr0, weight_decay, batch and returns best_params dict
    def objective(trial):
        lr0 = trial.suggest_float("lr0", 1e-5, 1e-2, log=True)
        weight_decay = trial.suggest_float("weight_decay", 1e-6, 1e-3, log=True)
        batch = trial.suggest_categorical("batch", [8, 16, 32])
        
        try:
            model = RTDETR(model_cfg['weights']) if 'rtdetr' in name.lower() else YOLO(model_cfg['weights'])
            model.train(
                data=str(YOLO_DIR / "config.yaml"),
                epochs=TUNE_EPOCHS,
                imgsz=IMG_SIZE,
                batch=batch,
                lr0=lr0,
                weight_decay=weight_decay,
                seed=SEED,
                verbose=False,
                plots=False,
                save=False,
                patience=TUNE_PATIENCE,
            )
            # evaluate on validation split
            val_metrics = model.val(split='val', verbose=False)
            # compute our scores 
            metrics = calculate_metrics(val_metrics)
            return metrics['f1']
        except Exception as e:
            print(f"Trial failed: {e}")
            return 0.0
    
    study = optuna.create_study(direction="maximize") 
    study.optimize(objective, n_trials=TUNE_ITERATIONS, show_progress_bar=False)
    return study.best_params

## Step 3: Train Base Model

In [6]:
results = []

for model_cfg in MODELS:
    name = model_cfg['name']
    print(f"\nTraining {name} (base)")
    
    model = RTDETR(model_cfg['weights']) if 'rtdetr' in name.lower() else YOLO(model_cfg['weights'])
    metrics = train_and_evaluate(model, name, f"{exp_id}_{name}_base")
    
    results.append({
        'model': name,
        'type': 'base',
        'val_precision': metrics['val']['precision'],
        'val_recall': metrics['val']['recall'],
        'val_f1': metrics['val']['f1'],
        'val_map50': metrics['val']['map50'],
        'test_precision': metrics['test']['precision'],
        'test_recall': metrics['test']['recall'],
        'test_f1':  metrics['test']['f1'],
        'test_map50': metrics['test']['map50'],
        'train_time': metrics['train_time'],
        'val_inference_time': metrics['val_inference_time'],
        'test_inference_time': metrics['test_inference_time']
    })
    
    print(f"{name}:  val_f1={metrics['val']['f1']:.4f}, test_f1={metrics['test']['f1']:.4f}, test_map50={metrics['test']['map50']:.4f}")


Training yolov8l (base)
New https://pypi.org/project/ultralytics/8.3.253 available 😃 Update with 'pip install -U ultralytics'
Ultralytics 8.3.250 🚀 Python-3.11.13 torch-2.9.1+cu128 CUDA:0 (NVIDIA GeForce RTX 4090 Laptop GPU, 15944MiB)
engine/trainer: agnostic_nms=False, amp=True, augment=False, auto_augment=randaugment, batch=16, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=/home/krschap/academia/dl4cv-object-detection-on-aerial-imagery/data/yolo/config.yaml, degrees=0.0, deterministic=True, device=None, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, epochs=200, erasing=0.4, exist_ok=False, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, half=False, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=256, int8=False, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.01, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=trai

/home/krschap/academia/dl4cv-object-detection-on-aerial-imagery/.venv/lib/python3.11/site-packages/torch/autograd/graph.py:841: UserWarning: grid_sampler_2d_backward_cuda does not have a deterministic implementation, but you set 'torch.use_deterministic_algorithms(True, warn_only=True)'. You can file an issue at https://github.com/pytorch/pytorch/issues to help us prioritize adding deterministic support for this operation. (Triggered internally at /pytorch/aten/src/ATen/Context.cpp:148.)
  return Variable._execution_engine.run_backward(  # Calls into the C++ engine to run the backward pass


      1/200      7.88G      1.874     0.4561     0.7449        344        256: 100% ━━━━━━━━━━━━ 20/20 2.6it/s 7.8s0.3s
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 3/3 2.4it/s 1.3s3.7s
                   all         89       2008      0.062      0.292     0.0438     0.0143

      Epoch    GPU_mem  giou_loss   cls_loss    l1_loss  Instances       Size
      2/200      5.58G      1.678     0.3144     0.4757        784        256: 0% ──────────── 0/20  0.2s

/home/krschap/academia/dl4cv-object-detection-on-aerial-imagery/.venv/lib/python3.11/site-packages/torch/autograd/graph.py:841: UserWarning: grid_sampler_2d_backward_cuda does not have a deterministic implementation, but you set 'torch.use_deterministic_algorithms(True, warn_only=True)'. You can file an issue at https://github.com/pytorch/pytorch/issues to help us prioritize adding deterministic support for this operation. (Triggered internally at /pytorch/aten/src/ATen/Context.cpp:148.)
  return Variable._execution_engine.run_backward(  # Calls into the C++ engine to run the backward pass


      2/200      6.83G      1.157     0.4861     0.2638        340        256: 100% ━━━━━━━━━━━━ 20/20 5.5it/s 3.7s0.2s
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 3/3 13.9it/s 0.2s.3s
                   all         89       2008      0.361      0.531      0.291     0.0963

      Epoch    GPU_mem  giou_loss   cls_loss    l1_loss  Instances       Size
      3/200      6.92G      0.935     0.5064      0.182        580        256: 0% ──────────── 0/20  0.2s

...

     93/200      7.82G     0.6333     0.4786    0.09832        304        256: 100% ━━━━━━━━━━━━ 20/20 5.7it/s 3.5s0.2s
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 3/3 13.0it/s 0.2s.3s
                   all         89       2008      0.709       0.75      0.654      0.279

      Epoch    GPU_mem  giou_loss   cls_loss    l1_loss  Instances       Size
     94/200      5.61G     0.6179     0.4682     0.1012        640        256: 5% ╸─────────── 1/20 1.8it/s 0.3s<10.6s

/home/krschap/academia/dl4cv-object-detection-on-aerial-imagery/.venv/lib/python3.11/site-packages/torch/autograd/graph.py:841: UserWarning: grid_sampler_2d_backward_cuda does not have a deterministic implementation, but you set 'torch.use_deterministic_algorithms(True, warn_only=True)'. You can file an issue at https://github.com/pytorch/pytorch/issues to help us prioritize adding deterministic support for this operation. (Triggered internally at /pytorch/aten/src/ATen/Context.cpp:148.)
  return Variable._execution_engine.run_backward(  # Calls into the C++ engine to run the backward pass


     94/200      7.96G     0.6283     0.4704    0.09203        238        256: 100% ━━━━━━━━━━━━ 20/20 5.6it/s 3.6s0.2s
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 3/3 13.8it/s 0.2s.3s
                   all         89       2008      0.716      0.778      0.672      0.316

      Epoch    GPU_mem  giou_loss   cls_loss    l1_loss  Instances       Size
     95/200      5.26G     0.6326     0.4519    0.09128        705        256: 5% ╸─────────── 1/20 1.8it/s 0.4s<10.8s

/home/krschap/academia/dl4cv-object-detection-on-aerial-imagery/.venv/lib/python3.11/site-packages/torch/autograd/graph.py:841: UserWarning: grid_sampler_2d_backward_cuda does not have a deterministic implementation, but you set 'torch.use_deterministic_algorithms(True, warn_only=True)'. You can file an issue at https://github.com/pytorch/pytorch/issues to help us prioritize adding deterministic support for this operation. (Triggered internally at /pytorch/aten/src/ATen/Context.cpp:148.)
  return Variable._execution_engine.run_backward(  # Calls into the C++ engine to run the backward pass


     95/200      7.69G     0.6383     0.4702     0.0952        336        256: 100% ━━━━━━━━━━━━ 20/20 5.5it/s 3.7s0.2s
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 3/3 12.7it/s 0.2s.3s
                   all         89       2008      0.703      0.758       0.65      0.258
EarlyStopping: Training stopped early as no improvement observed in last 30 epochs. Best results observed at epoch 65, best model saved as best.pt.
To update EarlyStopping(patience=30) pass a new patience value, i.e. `patience=300` or use `patience=0` to disable EarlyStopping.

95 epochs completed in 0.116 hours.
Optimizer stripped from /home/krschap/academia/dl4cv-object-detection-on-aerial-imagery/notebooks/runs/20260113_225000_rtdetr-l_base/weights/last.pt, 66.2MB
Optimizer stripped from /home/krschap/academia/dl4cv-object-detection-on-aerial-imagery/notebooks/runs/20260113_225000_rtdetr-l_base/weights/best.pt, 66.2MB

Validating /home/krschap/academi

## Step 4: Hyperparameter Tuning (Default)

In [7]:
if TUNE_DEFAULT:
    for model_cfg in MODELS:
        name = model_cfg['name']
        print(f"\nTuning {name} (ultralytics)")
        
        model = RTDETR(model_cfg['weights']) if 'rtdetr' in name.lower() else YOLO(model_cfg['weights'])
        
        model.tune(
            data=str(YOLO_DIR / "config.yaml"),
            epochs=TUNE_EPOCHS,
            iterations=TUNE_ITERATIONS,
            imgsz=IMG_SIZE,
            plots=False,
            save=False,
            val=True
        )
        
        best_cfg_path = Path(f"runs/detect/{name}/best_hyperparameters.yaml")
        tuned_params = {}
        if best_cfg_path.exists():
            with open(best_cfg_path, 'r') as f:
                tuned_params = yaml.safe_load(f)
            if 'close_mosaic' in tuned_params:
                tuned_params['close_mosaic'] = int(tuned_params['close_mosaic'])
        
        model = RTDETR(model_cfg['weights']) if 'rtdetr' in name.lower() else YOLO(model_cfg['weights'])
        metrics = train_and_evaluate(model, name, f"{exp_id}_{name}_default_tuned", tuned_params)
        
        results.append({
            'model': name,
            'type': 'default_tuned',
            'val_precision': metrics['val']['precision'],
            'val_recall': metrics['val']['recall'],
            'val_f1': metrics['val']['f1'],
            'val_map50':  metrics['val']['map50'],
            'test_precision': metrics['test']['precision'],
            'test_recall': metrics['test']['recall'],
            'test_f1':  metrics['test']['f1'],
            'test_map50': metrics['test']['map50'],
            'train_time': metrics['train_time'],
            'val_inference_time': metrics['val_inference_time'],
            'test_inference_time': metrics['test_inference_time']
        })
        
        exp_results_dir = RESULTS_DIR / exp_id
        exp_results_dir.mkdir(exist_ok=True)
        if best_cfg_path.exists():
            import shutil
            shutil.copy(best_cfg_path, exp_results_dir / f"{name}_default_best_hyperparameters.yaml")
        
        print(f"{name}: val_f1={metrics['val']['f1']:. 4f}, test_f1={metrics['test']['f1']:. 4f}, test_map50={metrics['test']['map50']:. 4f}")

## Step 5: Hyperparameter Tuning (Optuna)

In [ ]:
if TUNE_OPTUNA:
    for model_cfg in MODELS:
        name = model_cfg['name']
        print(f"\nTuning {name} (optuna)")
        
        best_params = tune_with_optuna(model_cfg, name)
        print(f"Best params: {best_params}")
        
        exp_results_dir = RESULTS_DIR / exp_id
        exp_results_dir. mkdir(exist_ok=True)
        with open(exp_results_dir / f"{name}_optuna_best_hyperparameters.yaml", 'w') as f:
            yaml.safe_dump(best_params, f)
        
        model = RTDETR(model_cfg['weights']) if 'rtdetr' in name.lower() else YOLO(model_cfg['weights'])
        metrics = train_and_evaluate(model, name, f"{exp_id}_{name}_optuna_tuned", best_params)
        
        results.append({
            'model': name,
            'type': 'optuna_tuned',
            'val_precision': metrics['val']['precision'],
            'val_recall': metrics['val']['recall'],
            'val_f1':  metrics['val']['f1'],
            'val_map50': metrics['val']['map50'],
            'test_precision': metrics['test']['precision'],
            'test_recall': metrics['test']['recall'],
            'test_f1': metrics['test']['f1'],
            'test_map50':  metrics['test']['map50'],
            'train_time': metrics['train_time'],
            'val_inference_time': metrics['val_inference_time'],
            'test_inference_time': metrics['test_inference_time']
        })
        
        print(f"{name}: val_f1={metrics['val']['f1']:.4f}, test_f1={metrics['test']['f1']:.4f}, test_map50={metrics['test']['map50']:.4f}")

[I 2026-01-13 23:02:52,904] A new study created in memory with name: no-name-b95e7875-6bea-48a9-ba2d-1d1368fc4ad7



Tuning yolov8l (optuna)
New https://pypi.org/project/ultralytics/8.3.253 available 😃 Update with 'pip install -U ultralytics'
Ultralytics 8.3.250 🚀 Python-3.11.13 torch-2.9.1+cu128 CUDA:0 (NVIDIA GeForce RTX 4090 Laptop GPU, 15944MiB)
engine/trainer: agnostic_nms=False, amp=True, augment=False, auto_augment=randaugment, batch=16, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=/home/krschap/academia/dl4cv-object-detection-on-aerial-imagery/data/yolo/config.yaml, degrees=0.0, deterministic=True, device=None, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, epochs=60, erasing=0.4, exist_ok=False, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, half=False, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=256, int8=False, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=9.674929409514124e-05, lrf=0.01, mask_ratio=4, max_det=300, mixu

[I 2026-01-13 23:03:11,348] Trial 0 finished with value: 0.0 and parameters: {'lr0': 9.674929409514124e-05, 'weight_decay': 1.6031527284832787e-05, 'batch': 16}. Best is trial 0 with value: 0.0.


Trial failed: [Errno 2] No such file or directory: '/home/krschap/academia/dl4cv-object-detection-on-aerial-imagery/notebooks/runs/detect/train/weights/last.pt'
New https://pypi.org/project/ultralytics/8.3.253 available 😃 Update with 'pip install -U ultralytics'
Ultralytics 8.3.250 🚀 Python-3.11.13 torch-2.9.1+cu128 CUDA:0 (NVIDIA GeForce RTX 4090 Laptop GPU, 15944MiB)
engine/trainer: agnostic_nms=False, amp=True, augment=False, auto_augment=randaugment, batch=8, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=/home/krschap/academia/dl4cv-object-detection-on-aerial-imagery/data/yolo/config.yaml, degrees=0.0, deterministic=True, device=None, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, epochs=60, erasing=0.4, exist_ok=False, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, half=False, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz

[I 2026-01-13 23:03:51,953] Trial 1 finished with value: 0.0 and parameters: {'lr0': 0.0018458115240370851, 'weight_decay': 2.5468828518680513e-05, 'batch': 8}. Best is trial 0 with value: 0.0.


Trial failed: [Errno 2] No such file or directory: '/home/krschap/academia/dl4cv-object-detection-on-aerial-imagery/notebooks/runs/detect/train2/weights/last.pt'
New https://pypi.org/project/ultralytics/8.3.253 available 😃 Update with 'pip install -U ultralytics'
Ultralytics 8.3.250 🚀 Python-3.11.13 torch-2.9.1+cu128 CUDA:0 (NVIDIA GeForce RTX 4090 Laptop GPU, 15944MiB)
engine/trainer: agnostic_nms=False, amp=True, augment=False, auto_augment=randaugment, batch=8, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=/home/krschap/academia/dl4cv-object-detection-on-aerial-imagery/data/yolo/config.yaml, degrees=0.0, deterministic=True, device=None, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, epochs=60, erasing=0.4, exist_ok=False, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, half=False, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgs

[I 2026-01-13 23:04:32,275] Trial 2 finished with value: 0.0 and parameters: {'lr0': 3.8854471671476925e-05, 'weight_decay': 2.6798009366940437e-06, 'batch': 8}. Best is trial 0 with value: 0.0.


Trial failed: [Errno 2] No such file or directory: '/home/krschap/academia/dl4cv-object-detection-on-aerial-imagery/notebooks/runs/detect/train3/weights/last.pt'
New https://pypi.org/project/ultralytics/8.3.253 available 😃 Update with 'pip install -U ultralytics'
Ultralytics 8.3.250 🚀 Python-3.11.13 torch-2.9.1+cu128 CUDA:0 (NVIDIA GeForce RTX 4090 Laptop GPU, 15944MiB)
engine/trainer: agnostic_nms=False, amp=True, augment=False, auto_augment=randaugment, batch=16, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=/home/krschap/academia/dl4cv-object-detection-on-aerial-imagery/data/yolo/config.yaml, degrees=0.0, deterministic=True, device=None, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, epochs=60, erasing=0.4, exist_ok=False, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, half=False, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, img

## Save Results

In [ ]:
import shutil

df = pd.DataFrame(results)

device_memory = torch.cuda.get_device_properties(0).total_memory / (1024.0 ** 3) if torch.cuda.is_available() else None

summary = {
    'exp_id': exp_id,
    'seed': SEED,
    'epochs': EPOCHS,
    'img_size': IMG_SIZE,
    'data' : label_stats,
    'batch': BATCH,
    'patience': PATIENCE,
    'tune_default': TUNE_DEFAULT,
    'tune_optuna': TUNE_OPTUNA,
    'tune_iterations': TUNE_ITERATIONS,
    'tune_epochs': TUNE_EPOCHS,
    'device': torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'CPU',
    'device_memory_gb': device_memory,
    'gpu_available': torch.cuda.is_available(),
    'models': [m['name'] for m in MODELS],
    'results': results,
    'best_model': results[df['test_map50'].idxmax()]['model'] if len(results) > 0 else None,
    'best_test_map50': float(df['test_map50'].max()) if len(results) > 0 else 0.0
}

exp_results_dir = RESULTS_DIR / exp_id
exp_results_dir.mkdir(exist_ok=True)

model_metrics = {}
RUN_DIR = Path("runs")

for model_cfg in MODELS:
    name = model_cfg['name']
    model = RTDETR(model_cfg['weights']) if 'rtdetr' in name.lower() else YOLO(model_cfg['weights'])
    
    total_params = sum(p.numel() for p in model.model.parameters())
    model_size = None
    
    for run_type in ['base', 'default_tuned', 'optuna_tuned']:
        run_name = f"{exp_id}_{name}_{run_type}"
        run_path = RUN_DIR / run_name
        
        if not run_path.exists():
            continue
        
        if model_size is None:
            best_pt = run_path / "weights" / "best.pt"
            if best_pt.exists():
                model_size = best_pt.stat().st_size / (1024.0 ** 2)
        
        model_results_dir = exp_results_dir / name / run_type
        model_results_dir.mkdir(parents=True, exist_ok=True)
        
        for file in ["results.csv","results.png","val_batch1_labels.jpg","val_batch1_pred.jpg","args.yaml"]:
            src = run_path / file
            if src.exists():
                shutil.copy(src, model_results_dir / file)
    
    model_metrics[name] = {
        'total_parameters': total_params,
        'model_size_mb': model_size
    }

summary['model_metrics'] = model_metrics

with open(exp_results_dir / "summary.json", 'w') as f:
    json.dump(summary, f, indent=2)

print("\nResults")
print(df[['model', 'type', 'val_f1', 'val_map50', 'test_f1', 'test_map50']].to_string(index=False))
print(f"\nBest: {summary['best_model']} (test_map50={summary['best_test_map50']:.4f})")
print(f"Saved: results/{exp_id}/summary.json")

## Step 6: Visualization

In [ ]:
from PIL import Image
import matplotlib.pyplot as plt

def show_plot(path, title=None):
    if not path.exists():
        return
    img = Image.open(path)
    plt.figure(figsize=(10, 6))
    plt.imshow(img)
    plt.axis("off")
    if title:
        plt.title(title)
    plt.show()

RUN_DIR = Path("runs")

for model_cfg in MODELS:
    name = model_cfg['name']
    
    for run_type in ['base', 'default_tuned', 'optuna_tuned']:
        run_name = f"{exp_id}_{name}_{run_type}"
        
        results_path = RUN_DIR / run_name / "results.png"
        if results_path.exists():
            show_plot(results_path, title=f"{name} ({run_type}) - Training Curves")
        
        cm_path = RUN_DIR / run_name / "confusion_matrix.png"
        if cm_path.exists():
            show_plot(cm_path, title=f"{name} ({run_type}) - Confusion Matrix")